In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Input,Dropout
from tensorflow.keras.models import Model
from sklearn.model_selection import KFold
from tensorflow.keras.preprocessing.image import load_img, img_to_array


In [ ]:
# Load sql data base as pandas data frame and create a preview
dataset_folder = '<DATASET_ROOT>/Exp_04_Vorversuche_YOLO/'
# Connect to sql database
sqlite_connection = sqlite3.connect('<DATASET_ROOT>/Exp_04_Vorversuche_YOLO/stripGen_results.db')
df = pd.read_sql_query("SELECT * FROM results", sqlite_connection)
df.head(10)

In [ ]:
# Convert absolute image paths to relative paths (relative to db file location)
df['image_path'] = df['image_path'].apply(lambda p: 'imgs/' + p.split('/')[-1])
# Add the path to the folder of the db file
image_paths = dataset_folder + df['image_path'].values
df['image_path'] = image_paths

In [ ]:
# Show one image
plt.imshow(X[0])
plt.title(f"Target: {y[0]}")
plt.axis('off')
plt.show()

## Density 2D Predictor

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='random_seed', y='strips_inside_box_count', data=df, palette='muted')
plt.xlabel('Seed')
plt.ylabel('Strips Inside Box Count')
plt.title('Distribution of Strips Inside Box Count for Each Seed')
plt.show()

In [ ]:
# Image regression with EfficientNet and k-fold cross-validation

# Parameters
IMG_SIZE = (225, 225)
BATCH_SIZE = 16
EPOCHS = 10
K_FOLDS = 5

# Prepare data
image_paths = df['image_path'].values
targets = df['density_2d'].values

def preprocess_image(path):
    img = load_img(path, target_size=IMG_SIZE)
    img = img_to_array(img) / 255.0
    return img

X = np.array([preprocess_image(p) for p in image_paths])
y = np.array(targets)

In [ ]:
fold_predictions = []
fold_truths = []

# K-Fold Cross Validation
kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
fold = 1
val_scores = []
for train_idx, val_idx in kf.split(X):
    print(f"Fold {fold}")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Build model
    base_model = EfficientNetB0(weights='imagenet', include_top=False, input_tensor=Input(shape=(*IMG_SIZE, 3)))
    x = GlobalAveragePooling2D()(base_model.output)
    # Add one dense layer
    x = Dense(128, activation='relu')(x)
    # Add a dropout layer
    x = Dropout(0.5)(x)
    output = Dense(1, activation='linear')(x)
    model = Model(inputs=base_model.input, outputs=output)

    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    # Train
    model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2)

    # Evaluate
    val_loss, val_mae = model.evaluate(X_val, y_val, verbose=0)
    print(f"Validation MAE: {val_mae}")
    val_scores.append(val_mae)
    # Predict on validation set
    y_pred = model.predict(X_val).flatten()
    fold_predictions.append(y_pred)
    fold_truths.append(y_val)
    fold += 1

print(f"Average Validation MAE over {K_FOLDS} folds: {np.mean(val_scores)}")

In [ ]:
#min,max density in dataset
min_density = df['density_2d'].min()
max_density = df['density_2d'].max()
var_density = df['density_2d'].var()
print(f"Min density_2d: {min_density}")
print(f"Max density_2d: {max_density}")
print(f"Var density_2d: {var_density}")
max_span = max_density - min_density
print(f"Max span density_2d: {max_span}")

In [ ]:
# Plotting
plt.figure(figsize=(8, 6))
colors = sns.color_palette('husl', K_FOLDS)
for i in range(K_FOLDS):
    plt.scatter([list(range(1, len(fold_truths[i]) + 1))],fold_predictions[i]-fold_truths[i], color=colors[i], alpha=0.7, label=f'Fold {i+1}')
# plot a horizontal line indicating min and max density
plt.axhline(y=max_span, color='red', linestyle='--', label='Density Span')
plt.axhline(y=-max_span, color='red', linestyle='--', label='Density Span')
plt.xlabel('Test Data Index')
plt.ylabel('Prediction Error (density_2d)')
#plt.title('Predicted vs Ground Truth (Stacked by Fold)')
plt.legend()
plt.show()

## Strip Count Predictor

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxenplot(x='random_seed', y='strips_inside_box_count', data=df, palette='muted')
plt.xlabel('Seed')
plt.ylabel('Strips Inside Box Count')
plt.title('Distribution of Strips Inside Box Count for Each Seed')
plt.show()

In [ ]:
# Image regression for strip count using EfficientNet and k-fold cross-validation
IMG_SIZE = (512, 512)

# Filter out images with strip count less than 60
# df_filtered = df[df['strips_inside_box_count'] >= 60]


# Prepare data for new target
image_paths = df['image_path'].values
targets = df['strips_inside_box_count'].values

# Min-max normalization for targets
target_min = targets.min()
target_max = targets.max()
targets_norm = (targets - target_min) / (target_max - target_min)

def preprocess_image(path):
    img = load_img(path, target_size=IMG_SIZE)
    img = img_to_array(img) / 255.0
    return img

X = np.array([preprocess_image(p) for p in image_paths])
y = np.array(targets_norm)


In [ ]:
# plot a histogram of the target value
plt.figure(figsize=(8, 6))
plt.hist(y.flatten(), bins=50, color='blue', alpha=0.7)
plt.xlabel('Target Value (Count)')
plt.ylabel('Frequency')
plt.title('Histogram of Target Values')
plt.show()

In [ ]:
BATCH_SIZE = 16
EPOCHS = 30
K_FOLDS = 3

### Simple CNN Architecture for Regression Task

In [ ]:
fold_predictions = []
fold_truths = []
fold_train_losses = []

kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
fold = 1
val_scores = []
for train_idx, val_idx in kf.split(X):
    print(f"Fold {fold}")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]

    # Build model with ResNet50 backbone
    from tensorflow.keras.applications import ResNet50
    base_model = ResNet50(weights='imagenet', include_top=False, input_tensor=Input(shape=(*IMG_SIZE, 3)))
    base_model.trainable = False  # Freeze backbone weights
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(128, activation='relu')(x)
    x = Dropout(0.5)(x)
    output = Dense(1, activation='linear')(x)
    model = Model(inputs=base_model.input, outputs=output)

    model.compile(optimizer='adam', loss='mse', metrics=['mae'])

    # Train
    history=model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2)

    # Evaluate
    val_loss, val_mae = model.evaluate(X_val, y_val, verbose=0)
    print(f"Validation MAE: {val_mae}")
    val_scores.append(val_mae)
    # Predict on validation set
    y_pred = model.predict(X_val).flatten()
    # Inverse normalization
    y_pred_inv = y_pred * (target_max - target_min) + target_min
    y_val_inv = y_val * (target_max - target_min) + target_min
    fold_predictions.append(y_pred_inv)
    fold_truths.append(y_val_inv)
    fold_train_losses.append(history.history['loss'])
    fold += 1

print(f"Average Validation MAE over {K_FOLDS} folds: {np.mean(val_scores)}")


In [ ]:
# plot the training loss
plt.figure(figsize=(8, 6))
colors = sns.color_palette('tab10', K_FOLDS)
for i in range(K_FOLDS):
    plt.plot(fold_train_losses[i], color=colors[i], alpha=0.7, label=f'Fold {i+1}')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss by Fold')
plt.legend()
plt.show()

In [ ]:
# Visualization
plt.figure(figsize=(8, 6))
colors = sns.color_palette('tab10', K_FOLDS)
for i in range(K_FOLDS):
    indices = list(range(1, len(fold_truths[i]) + 1))
    plt.scatter(fold_truths[i],fold_predictions[i], color=colors[i], alpha=0.7, label=f'Fold {i+1}')
plt.xlabel('Ground Truth')
plt.ylabel('Prediction Error (strips_inside_box_count)')
plt.legend()
plt.show()

## Density Mapping Approach

In [ ]:
# CSRNet implementation for image-based counting
from tensorflow.keras.layers import Conv2D, BatchNormalization, Activation, MaxPooling2D, ZeroPadding2D
from tensorflow.keras.layers import Input, Add, UpSampling2D, concatenate
from tensorflow.keras.models import Model

# CSRNet backbone: VGG-like front-end, dilated conv back-end

def csrnet(input_shape):
    inputs = Input(shape=input_shape)
    # Front-end (VGG-like)
    x = Conv2D(64, 3, padding='same', activation='relu')(inputs)
    x = Conv2D(64, 3, padding='same', activation='relu')(x)
    x = MaxPooling2D(2)(x)
    x = Conv2D(128, 3, padding='same', activation='relu')(x)
    x = Conv2D(128, 3, padding='same', activation='relu')(x)
    x = MaxPooling2D(2)(x)
    x = Conv2D(256, 3, padding='same', activation='relu')(x)
    x = Conv2D(256, 3, padding='same', activation='relu')(x)
    x = Conv2D(256, 3, padding='same', activation='relu')(x)
    x = MaxPooling2D(2)(x)
    # Back-end (dilated convs)
    x = Conv2D(512, 3, dilation_rate=2, padding='same', activation='relu')(x)
    x = Conv2D(512, 3, dilation_rate=2, padding='same', activation='relu')(x)
    x = Conv2D(512, 3, dilation_rate=2, padding='same', activation='relu')(x)
    x = Conv2D(256, 3, dilation_rate=2, padding='same', activation='relu')(x)
    x = Conv2D(128, 3, dilation_rate=2, padding='same', activation='relu')(x)
    x = Conv2D(64, 3, dilation_rate=2, padding='same', activation='relu')(x)
    # Output: density map (regression)
    density_map = Conv2D(1, 1, padding='same', activation='relu')(x)
    # For counting, use global average pooling to get a single value
    from tensorflow.keras.layers import GlobalAveragePooling2D
    count = GlobalAveragePooling2D()(density_map)
    model = Model(inputs, count)
    return model

# Prepare data shape
input_shape = (*IMG_SIZE, 3)
csrnet_model = csrnet(input_shape)
csrnet_model.compile(optimizer='adam', loss='mse', metrics=['mae'])

# Train CSRNet (predicting density map, but for regression, use sum as count)
fold_predictions = []
fold_truths = []
fold_train_losses = []
kf = KFold(n_splits=K_FOLDS, shuffle=True, random_state=42)
fold = 1
val_scores = []
for train_idx, val_idx in kf.split(X):
    print(f"CSRNet Fold {fold}")
    X_train, X_val = X[train_idx], X[val_idx]
    y_train, y_val = y[train_idx], y[val_idx]
    # For CSRNet, target is still normalized count (not density map)
    # Expand dims for regression output
    y_train = y_train.reshape(-1, 1)
    y_val = y_val.reshape(-1, 1)
    history = csrnet_model.fit(X_train, y_train, validation_data=(X_val, y_val), epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=2)
    val_loss, val_mae = csrnet_model.evaluate(X_val, y_val, verbose=0)
    print(f"Validation MAE: {val_mae}")
    val_scores.append(val_mae)
    y_pred = csrnet_model.predict(X_val).flatten()
    y_pred_inv = y_pred * (target_max - target_min) + target_min
    y_val_inv = y_val.flatten() * (target_max - target_min) + target_min
    fold_predictions.append(y_pred_inv)
    fold_truths.append(y_val_inv)
    fold_train_losses.append(history.history['loss'])
    fold += 1
print(f"CSRNet Average Validation MAE over {K_FOLDS} folds: {np.mean(val_scores)}")


In [ ]:
# plot the training loss
plt.figure(figsize=(8, 6))
colors = sns.color_palette('tab10', K_FOLDS)
for i in range(K_FOLDS):
    plt.plot(fold_train_losses[i], color=colors[i], alpha=0.7, label=f'Fold {i+1}')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training Loss by Fold')
plt.legend()
plt.show()

In [ ]:
# Visualization
plt.figure(figsize=(8, 6))
colors = sns.color_palette('tab10', K_FOLDS)
for i in range(K_FOLDS):
    indices = list(range(1, len(fold_truths[i]) + 1))
    plt.scatter(fold_truths[i],fold_predictions[i], color=colors[i], alpha=0.7, label=f'Fold {i+1}')
plt.xlabel('Ground Truth')
plt.ylabel('Prediction Error (strips_inside_box_count)')
plt.legend()
plt.show()